# RGCA Baseline Experiment Notebook

This notebook runs the structured pre-RGCA baseline experiments for retrieval-induced hallucination in chest X-ray report generation.

Baseline pipeline:

```text
Image/study record -> Retriever -> Retrieved reports -> Generator -> Generated report -> Evaluation
```

The purpose is not to beat SOTA yet. The purpose is to create a reproducible experiment setup that shows retrieval behavior, mismatch behavior, and the evaluation artifacts needed before implementing RGCA.


## What This Notebook Produces

If run end-to-end, this notebook produces:

- a MIMIC pilot subset JSONL, if raw MIMIC files are available
- a structured experiment suite with named runs
- retrieval outputs for clean and mismatch retrieval
- generated reports for no-retrieval, retrieval, and mismatch modes
- hallucination evaluation summaries
- CSV/Markdown result tables
- a private zip artifact for download

Important: MIMIC-CXR data and derived report text must remain private. Do not publish raw reports, images, or generated outputs containing report text to GitHub.


## 1. Environment Setup

Run this cell first. It detects whether we are in Kaggle or a local checkout and sets paths accordingly.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

IS_KAGGLE = Path('/kaggle/working').exists()

USE_DEMO_DATA = not IS_KAGGLE

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/RGCA')
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROJECT_ROOT exists:', PROJECT_ROOT.exists())


## 2. Clone Or Update The Repository On Kaggle

Run this only on Kaggle. If you are local, skip it.


In [ ]:
if IS_KAGGLE:
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/pidoxy/RGCA.git', str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
else:
    print('Local environment detected. Skipping clone/pull/install cell.')


## 3. Import RGCA Helpers


In [ ]:
from rgca_baseline.io_utils import read_jsonl
from rgca_baseline.pipeline import load_studies

def run_command(command, cwd=PROJECT_ROOT):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, cwd=str(cwd), check=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def print_json(payload):
    print(json.dumps(payload, indent=2))

print('Imports ready')


## 4. Restore Data From GCS Or PhysioNet

Use this section when Kaggle shows **No input attached**. The recommended path is:

1. Download only the small MIMIC-CXR-JPG metadata files from the GCS bucket.
2. Restore the MIMIC-CXR reports zip from either a reports GCS bucket or PhysioNet.
3. Build `mimic_subset.jsonl`.
4. Package that subset as a private Kaggle dataset so future notebook runs do not need to redownload raw files.

The GCS console URL is for the browser. In code we use the `gs://` path.


In [ ]:
# Data restore settings.
# Change these toggles deliberately, then run the restore cells below.

GCS_JPG_BUCKET = "gs://mimic-cxr-jpg-2.1.0.physionet.org"

# Optional. Use only if you also have access to a MIMIC-CXR reports bucket.
# If you do not have this bucket, leave it empty and use the PhysioNet reports option below.
GCS_REPORTS_BUCKET = ""  # example: "gs://mimic-cxr-2.1.0.physionet.org"

DATA_WORK_DIR = Path("/kaggle/working/physionet") if IS_KAGGLE else PROJECT_ROOT / "data" / "raw"

RESTORE_SMALL_FILES_FROM_GCS = False
RESTORE_REPORTS_FROM_GCS = False
DOWNLOAD_REPORTS_FROM_PHYSIONET = False
COPY_PILOT_IMAGES_FROM_GCS = False

PHYSIONET_USERNAME = ""  # fill only if DOWNLOAD_REPORTS_FROM_PHYSIONET = True

print("DATA_WORK_DIR:", DATA_WORK_DIR)
print("GCS_JPG_BUCKET:", GCS_JPG_BUCKET)
print("GCS_REPORTS_BUCKET:", GCS_REPORTS_BUCKET or "not configured")
print("RESTORE_SMALL_FILES_FROM_GCS:", RESTORE_SMALL_FILES_FROM_GCS)
print("RESTORE_REPORTS_FROM_GCS:", RESTORE_REPORTS_FROM_GCS)
print("DOWNLOAD_REPORTS_FROM_PHYSIONET:", DOWNLOAD_REPORTS_FROM_PHYSIONET)
print("COPY_PILOT_IMAGES_FROM_GCS:", COPY_PILOT_IMAGES_FROM_GCS)


### 4A. Check GCS Access

Run this before downloading. If it fails, make sure your Kaggle notebook has internet enabled and that your Google Cloud credentials/session can access the bucket.


In [ ]:
if IS_KAGGLE:
    run_command(["gcloud", "--version"])
    run_command(["gcloud", "storage", "ls", GCS_JPG_BUCKET])
else:
    print("Skipping GCS check outside Kaggle.")


### 4B. Download Small MIMIC-CXR-JPG Metadata From GCS

This downloads only the small metadata files needed to plan/build the pilot subset. It does **not** download the full 500GB+ JPG archive.

Set `RESTORE_SMALL_FILES_FROM_GCS = True` in the settings cell above before running this cell.


In [ ]:
if IS_KAGGLE and RESTORE_SMALL_FILES_FROM_GCS:
    DATA_WORK_DIR.mkdir(parents=True, exist_ok=True)
    for filename in [
        "mimic-cxr-2.0.0-metadata.csv.gz",
        "mimic-cxr-2.0.0-split.csv.gz",
        "mimic-cxr-2.0.0-chexpert.csv.gz",
    ]:
        destination = DATA_WORK_DIR / filename
        if destination.exists():
            print("Already exists:", destination)
            continue
        run_command(["gcloud", "storage", "cp", f"{GCS_JPG_BUCKET.rstrip('/')}/{filename}", str(destination)])
else:
    print("Skipping small-file restore. Set RESTORE_SMALL_FILES_FROM_GCS = True to run it.")


### 4C. Restore MIMIC-CXR Reports

The JPG bucket does not contain the free-text reports. For report generation experiments, we need `mimic-cxr-reports.zip` from MIMIC-CXR.

Use one of these options:

- If you have a reports GCS bucket, set `GCS_REPORTS_BUCKET` and `RESTORE_REPORTS_FROM_GCS = True`.
- Otherwise set `DOWNLOAD_REPORTS_FROM_PHYSIONET = True` and provide your PhysioNet username when prompted.

This downloads the reports zip only, not the full DICOM dataset.


In [ ]:
reports_zip = DATA_WORK_DIR / "mimic-cxr-reports.zip"
reports_extract_dir = DATA_WORK_DIR / "mimic-cxr"
reports_files_dir = reports_extract_dir / "files"

if IS_KAGGLE and RESTORE_REPORTS_FROM_GCS:
    if not GCS_REPORTS_BUCKET:
        raise ValueError("Set GCS_REPORTS_BUCKET before using RESTORE_REPORTS_FROM_GCS.")
    DATA_WORK_DIR.mkdir(parents=True, exist_ok=True)
    if not reports_zip.exists():
        run_command(["gcloud", "storage", "cp", f"{GCS_REPORTS_BUCKET.rstrip('/')}/mimic-cxr-reports.zip", str(reports_zip)])
    else:
        print("Reports zip already exists:", reports_zip)

elif IS_KAGGLE and DOWNLOAD_REPORTS_FROM_PHYSIONET:
    import getpass
    if not PHYSIONET_USERNAME:
        raise ValueError("Set PHYSIONET_USERNAME before downloading from PhysioNet.")
    DATA_WORK_DIR.mkdir(parents=True, exist_ok=True)
    password = getpass.getpass("PhysioNet password: ")
    netrc_path = Path.home() / ".netrc"
    netrc_path.write_text(f"machine physionet.org login {PHYSIONET_USERNAME} password {password}
", encoding="utf-8")
    netrc_path.chmod(0o600)
    run_command([
        "wget", "-c", "--netrc",
        "https://physionet.org/files/mimic-cxr/2.1.0/mimic-cxr-reports.zip",
        "-O", str(reports_zip),
    ])
    netrc_path.unlink(missing_ok=True)
else:
    print("Skipping reports restore. Configure GCS reports or PhysioNet download if reports are missing.")

if reports_zip.exists() and not reports_files_dir.exists():
    reports_extract_dir.mkdir(parents=True, exist_ok=True)
    run_command(["unzip", "-q", str(reports_zip), "-d", str(reports_extract_dir)])

print("reports_zip:", reports_zip, "exists=", reports_zip.exists())
print("reports_files_dir:", reports_files_dir, "exists=", reports_files_dir.exists())


## 4. Configure Data Paths

These defaults match the paths we have been using in Kaggle. Adjust them if your uploaded dataset is attached under `/kaggle/input/...` instead of `/kaggle/working/physionet/...`.


In [ ]:
def first_existing(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return Path(candidates[0])

def find_first_file(filename, roots):
    candidates = []
    for root in roots:
        root = Path(root)
        if root.exists():
            candidates.extend(root.rglob(filename))
    return sorted(candidates)[0] if candidates else Path(roots[0]) / filename

def find_optional_file(filename, roots):
    candidates = []
    for root in roots:
        root = Path(root)
        if root.exists():
            candidates.extend(root.rglob(filename))
    return sorted(candidates)[0] if candidates else None

def find_dataset_files_dir(dataset_name, roots):
    direct_candidates = []
    for root in roots:
        root = Path(root)
        direct_candidates.extend([
            root / dataset_name / 'files',
            root / dataset_name / '2.1.0' / 'files',
            root / dataset_name / '2.0.0' / 'files',
        ])
    existing = [path for path in direct_candidates if path.exists()]
    if existing:
        return sorted(existing)[0]

    discovered = []
    for root in roots:
        root = Path(root)
        if root.exists():
            discovered.extend(path / 'files' for path in root.rglob(dataset_name) if (path / 'files').exists())
    return sorted(discovered)[0] if discovered else Path(roots[0]) / dataset_name / 'files'

if IS_KAGGLE:
    SEARCH_ROOTS = [Path('/kaggle/working'), Path('/kaggle/input')]
    PHYSIONET_ROOT = first_existing([DATA_WORK_DIR, Path('/kaggle/input')])
    MIMIC_REPORTS_ROOT = find_dataset_files_dir('mimic-cxr', SEARCH_ROOTS)
    MIMIC_JPG_ROOT = find_dataset_files_dir('mimic-cxr-jpg', SEARCH_ROOTS)
    METADATA_PATH = find_first_file('mimic-cxr-2.0.0-metadata.csv.gz', SEARCH_ROOTS)
    SPLIT_PATH = find_first_file('mimic-cxr-2.0.0-split.csv.gz', SEARCH_ROOTS)
    LABELS_PATH = find_first_file('mimic-cxr-2.0.0-chexpert.csv.gz', SEARCH_ROOTS)
    PILOT_OUTPUT_DIR = Path('/kaggle/working/rgca_pilot_500')
    SUITE_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0')
else:
    PHYSIONET_ROOT = PROJECT_ROOT / 'data' / 'raw'
    MIMIC_REPORTS_ROOT = PHYSIONET_ROOT / 'mimic-cxr' / 'files'
    MIMIC_JPG_ROOT = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'files'
    METADATA_PATH = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'mimic-cxr-2.0.0-metadata.csv.gz'
    SPLIT_PATH = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'mimic-cxr-2.0.0-split.csv.gz'
    LABELS_PATH = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'mimic-cxr-2.0.0-chexpert.csv.gz'
    PILOT_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'rgca_pilot_500'
    SUITE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'rgca_experiments' / 'notebook_demo_suite'

ATTACHED_SUBSET_PATH = find_optional_file('mimic_subset.jsonl', [Path('/kaggle/input')]) if IS_KAGGLE else None
SUBSET_PATH = PROJECT_ROOT / 'data' / 'demo' / 'demo_studies.jsonl' if USE_DEMO_DATA else (ATTACHED_SUBSET_PATH or PILOT_OUTPUT_DIR / 'data' / 'mimic_subset.jsonl')
SUITE_CONFIG = PROJECT_ROOT / 'configs' / 'mimic_pilot_suite.json'

paths = {
    'METADATA_PATH': METADATA_PATH,
    'SPLIT_PATH': SPLIT_PATH,
    'LABELS_PATH': LABELS_PATH,
    'MIMIC_REPORTS_ROOT': MIMIC_REPORTS_ROOT,
    'MIMIC_JPG_ROOT': MIMIC_JPG_ROOT,
    'ATTACHED_SUBSET_PATH': ATTACHED_SUBSET_PATH,
    'SUBSET_PATH': SUBSET_PATH,
    'SUITE_CONFIG': SUITE_CONFIG,
    'SUITE_OUTPUT_DIR': SUITE_OUTPUT_DIR,
}

print('USE_DEMO_DATA:', USE_DEMO_DATA)
for name, path in paths.items():
    print(f'{name}: {path} | exists={Path(path).exists()}')


## 5. Choose Pilot Size

Use 500 studies for the current working pilot. This creates 400 retrieval-pool studies and 100 evaluation studies.

For a quick smoke test, change these to `RETRIEVAL_LIMIT = 80` and `EVAL_LIMIT = 20`.


In [ ]:
RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100
TOP_K = 3

print('retrieval_limit:', RETRIEVAL_LIMIT)
print('eval_limit:', EVAL_LIMIT)
print('top_k:', TOP_K)


## 6. Build The Pilot Subset

Run this if `SUBSET_PATH` does not already exist. The output is a study-level JSONL with image/report pairing and train/validate split mapping.


In [ ]:
if USE_DEMO_DATA:
    print('Using demo dataset for local smoke testing:', SUBSET_PATH)
elif SUBSET_PATH.exists():
    print('Subset already exists:', SUBSET_PATH)
else:
    required_paths = {
        'metadata': METADATA_PATH,
        'split': SPLIT_PATH,
        'labels': LABELS_PATH,
        'reports_root': MIMIC_REPORTS_ROOT,
        'images_root': MIMIC_JPG_ROOT,
    }
    missing = {name: str(path) for name, path in required_paths.items() if not Path(path).exists()}
    if missing:
        raise FileNotFoundError(
            'Cannot build the MIMIC pilot subset because required files/folders are missing. '
            'Either rerun the PhysioNet download cells, attach a private Kaggle dataset, '
            'or point the path variables in Section 4 to the correct locations. Missing: '
            + json.dumps(missing, indent=2)
        )
    command = [
        sys.executable,
        'scripts/kaggle_run_pilot.py',
        '--metadata', str(METADATA_PATH),
        '--split', str(SPLIT_PATH),
        '--labels', str(LABELS_PATH),
        '--reports-root', str(MIMIC_REPORTS_ROOT),
        '--images-root', str(MIMIC_JPG_ROOT),
        '--output-dir', str(PILOT_OUTPUT_DIR),
        '--limit', str(RETRIEVAL_LIMIT + EVAL_LIMIT),
        '--retrieval-limit', str(RETRIEVAL_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--retriever', 'lexical',
        '--generator', 'mock',
        '--top-k', str(TOP_K),
    ]
    run_command(command)

print('Subset path:', SUBSET_PATH)
print('Subset exists:', SUBSET_PATH.exists())


## 7. Validate The Subset

This confirms that we have both retrieval-pool and evaluation records. If `eval` is zero, the experiment is invalid.


In [ ]:
studies = load_studies(SUBSET_PATH)
retrieval_pool = [study for study in studies if study.split == 'retrieval_pool']
eval_studies = [study for study in studies if study.split == 'eval']

print('total_studies:', len(studies))
print('retrieval_pool:', len(retrieval_pool))
print('eval_studies:', len(eval_studies))
print('first_record:', studies[0].to_dict() if studies else None)

assert retrieval_pool, 'Invalid subset: no retrieval_pool records found.'
assert eval_studies, 'Invalid subset: no eval records found.'


## 8. Optional: Copy Pilot Images From GCS

The current mock and stress experiments do not open image pixels, so `mimic_subset.jsonl` is enough for baseline validation. For the next real VLM phase, copy only the pilot images from the JPG bucket instead of downloading the full image archive.

Set `COPY_PILOT_IMAGES_FROM_GCS = True` in Section 4 before running this cell.


In [ ]:
PILOT_IMAGES_ROOT = Path('/kaggle/working/rgca_private_dataset/mimic-cxr-jpg/files') if IS_KAGGLE else PROJECT_ROOT / 'outputs' / 'rgca_private_dataset' / 'mimic-cxr-jpg' / 'files'

if IS_KAGGLE and COPY_PILOT_IMAGES_FROM_GCS:
    run_command([
        sys.executable,
        'scripts/copy_pilot_images_from_gcs.py',
        '--subset-jsonl', str(SUBSET_PATH),
        '--images-root', str(MIMIC_JPG_ROOT),
        '--gcs-bucket', GCS_JPG_BUCKET,
        '--output-root', str(PILOT_IMAGES_ROOT),
    ])
else:
    print('Skipping pilot image copy. Set COPY_PILOT_IMAGES_FROM_GCS = True when you are ready for real VLM image inputs.')

print('PILOT_IMAGES_ROOT:', PILOT_IMAGES_ROOT, 'exists=', PILOT_IMAGES_ROOT.exists())


## 9. Run The Structured Experiment Suite

This is now the preferred path. It runs the planned experiment matrix from `configs/mimic_pilot_suite.json` and writes one output folder per experiment.


In [ ]:
run_command([
    sys.executable,
    'scripts/run_experiment_suite.py',
    '--config', str(SUITE_CONFIG),
    '--input', str(SUBSET_PATH),
    '--output-dir', str(SUITE_OUTPUT_DIR),
    '--overwrite',
])


## 10. Summarize Results Into Tables


In [ ]:
TABLES_DIR = SUITE_OUTPUT_DIR / 'tables'
SUITE_MANIFEST = SUITE_OUTPUT_DIR / 'suite_manifest.json'

run_command([
    sys.executable,
    'scripts/summarize_experiment_suite.py',
    '--manifest', str(SUITE_MANIFEST),
    '--output-dir', str(TABLES_DIR),
])

print('Markdown table:', TABLES_DIR / 'suite_summary.md')
print('CSV table:', TABLES_DIR / 'suite_summary.csv')


## 11. Display The Summary Table


In [ ]:
summary_md = (TABLES_DIR / 'suite_summary.md').read_text(encoding='utf-8')
print(summary_md[:6000])


## 12. Inspect The Suite Manifest

The manifest is the reproducibility record. It stores the subset path, output path, and settings/results for every named experiment.


In [ ]:
suite_manifest = read_json(SUITE_MANIFEST)
print('suite_name:', suite_manifest['suite_name'])
print('input_path:', suite_manifest['input_path'])
print('num_experiments:', len(suite_manifest['experiments']))

for experiment in suite_manifest['experiments']:
    pipeline = experiment['pipeline_summary']
    print({
        'name': experiment['name'],
        'retriever': pipeline['retriever_backend'],
        'generator': pipeline['generator_backend'],
        'top_k': pipeline['top_k'],
        'generated_counts': pipeline['generated_counts'],
    })


## 13. Inspect Mismatch Examples

Use this to manually review examples where retrieved labels appear in generated labels. Start with the controlled stress experiment, then later repeat for a real VLM experiment.


In [ ]:
EXPERIMENT_TO_INSPECT = 'E02_stress_lexical_k3'
DETAILS_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'evaluation' / 'mismatch' / 'evaluation_details.jsonl'
GENERATIONS_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'baseline' / 'generations_mismatch.jsonl'
RETRIEVAL_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'baseline' / 'mismatch_results.jsonl'

details = read_jsonl(DETAILS_PATH)
generations = {row['study_id']: row for row in read_jsonl(GENERATIONS_PATH)}
retrievals = {row['target_study']: row for row in read_jsonl(RETRIEVAL_PATH)}

interesting = [row for row in details if row['retrieval_induced_flags']]
print('cases_with_retrieval_induced_flags:', len(interesting))

for row in interesting[:5]:
    study_id = row['study_id']
    print('\n' + '=' * 100)
    print('study_id:', study_id)
    print('reference_labels:', row['reference_labels'])
    print('retrieved_labels:', row['retrieved_labels'])
    print('generated_labels:', row['generated_labels'])
    print('retrieval_induced_flags:', row['retrieval_induced_flags'])
    print('\nGenerated report snippet:')
    print(generations[study_id]['generated_report'][:1500])
    print('\nFirst retrieved report snippet:')
    print(retrievals[study_id]['retrieved_reports'][0][:1500])


## 14. Compare `k = 1, 3, 5`

This lets us quickly check whether increasing retrieved context increases copied unsupported findings in the controlled stress setup.


In [ ]:
for experiment in suite_manifest['experiments']:
    name = experiment['name']
    if not name.startswith('E0') or 'stress_lexical' not in name:
        continue
    pipeline = experiment['pipeline_summary']
    mismatch = experiment['evaluation_summaries']['mismatch']['mismatch']
    print({
        'experiment': name,
        'top_k': pipeline['top_k'],
        'retrieval_induced_hallucination_rate': mismatch['retrieval_induced_hallucination_rate'],
        'retrieval_copy_rate': mismatch['retrieval_copy_rate'],
        'total_retrieval_induced_hallucinations': mismatch['total_retrieval_induced_hallucinations'],
    })


## 15. Package A Private Kaggle Dataset Folder

This creates `/kaggle/working/rgca_private_dataset.zip`.

To reuse it in future Kaggle sessions:

1. Download the zip from the notebook output, or use Kaggle's output browser.
2. Go to Kaggle Datasets -> New Dataset.
3. Upload the zip contents or the zip file.
4. Set visibility to **Private**.
5. In a new notebook, click **Add Input** and attach that private dataset.
6. Rerun from Section 1. The notebook will auto-detect `mimic_subset.jsonl` under `/kaggle/input`.

Keep this dataset private because it contains MIMIC-derived clinical text.


In [ ]:
PRIVATE_DATASET_DIR = Path('/kaggle/working/rgca_private_dataset') if IS_KAGGLE else PROJECT_ROOT / 'outputs' / 'rgca_private_dataset'
PRIVATE_DATASET_ZIP = PRIVATE_DATASET_DIR.with_suffix('.zip')

package_command = [
    sys.executable,
    'scripts/package_private_kaggle_dataset.py',
    '--subset-jsonl', str(SUBSET_PATH),
    '--output-dir', str(PRIVATE_DATASET_DIR),
    '--suite-output-dir', str(SUITE_OUTPUT_DIR),
    '--zip',
]

if PILOT_IMAGES_ROOT.exists():
    package_command.extend(['--images-root', str(PILOT_IMAGES_ROOT)])

run_command(package_command)

print('Private dataset folder:', PRIVATE_DATASET_DIR)
print('Private dataset zip:', PRIVATE_DATASET_ZIP)
print('Zip exists:', PRIVATE_DATASET_ZIP.exists())
if PRIVATE_DATASET_ZIP.exists():
    print('Size MB:', round(PRIVATE_DATASET_ZIP.stat().st_size / (1024 * 1024), 2))


## 16. Optional: Run A Single Experiment

Use this when debugging one run without rerunning the entire suite.


In [ ]:
# Example only. Uncomment to run a single experiment.
# run_command([
#     sys.executable,
#     'scripts/run_experiment_suite.py',
#     '--config', str(SUITE_CONFIG),
#     '--input', str(SUBSET_PATH),
#     '--output-dir', str(SUITE_OUTPUT_DIR),
#     '--only', 'E02_stress_lexical_k3',
#     '--overwrite',
# ])


## 17. How To Interpret Current Results

Use this language carefully:

- `mock` generator runs validate infrastructure only.
- `retrieval_copy_stress` runs validate the mismatch/evaluation protocol under controlled contamination.
- These are not final clinical results.
- The paper-level empirical claim requires real image/text retrieval and a real VLM generator.

Current evidence tier:

```text
Tier 1: infrastructure evidence      -> complete
Tier 2: controlled failure evidence  -> complete after stress suite
Tier 3: real VLM evidence            -> next
```


## 18. Next Implementation Steps

After this notebook runs successfully:

1. Save the private zip artifact.
2. Use `suite_summary.csv` for early tables.
3. Manually review 10 to 20 mismatch cases.
4. Implement real image/text retrieval, ideally BioMedCLIP-style retrieval.
5. Add a real VLM backend and rerun the same structured suite.

This keeps us disciplined: same subset, same matrix, same output contract, stronger backends over time.
